In [ ]:
#%pip3 install seaborn
#%pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
#%pip install torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu118

from PIL import Image
import torch
import torch.nn as nn
from torch import nn, save, load
from torch.optim import Adam
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

import matplotlib.pyplot as plt
#import seaborn as sns

import pandas as pd
import numpy as np
import sqlite3

#from sklearn.metrics import mean_squared_error

from Sequences import Sequences
from ValueSequencer import HexagramValueSequencer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)


In [ ]:
"""
class EegNN(nn.Module):
	def __init__(self):
		super(EegNN, self).__init__()
		self.input_layer = nn.Linear(20, 100)
		#self.layer_1 = nn.Linear(100, 1)
		self.layer_1 = nn.Linear(100, 20)
		self.layer_2 = nn.Linear(20, 10)
		self.layer_3 = nn.Linear(10, 1)
		# Sigmoid function for output
		self.sigmoid = nn.Sigmoid()

	def forward(self, x):
		x = torch.relu(self.input_layer(x))
		#x = self.layer_1(x)
		x = torch.relu(self.layer_1(x))
		x = torch.relu(self.layer_2(x))
		x = self.layer_3(x)
		x = self.sigmoid(x)
		return x

		
class EegNN(nn.Module):
	def __init__(self):
		super(EegNN, self).__init__()
		self.model = nn.Sequential(
			nn.Linear(20, 100),
			nn.ReLU(),
			nn.Linear(100, 20),
			nn.ReLU(),
			nn.Linear(20, 10),
			nn.ReLU(),
			nn.Linear(10, 1),
			nn.Sigmoid()
			)

	def forward(self, x):
		return self.model(x)

class EegNN(nn.Module):
	def __init__(self):
		super(EegNN, self).__init__()
		self.input_layer = nn.Linear(20, 100, dtype=torch.float32)
		
		#self.layer_1 = nn.Linear(100, 1, dtype=torch.float32)

		self.layer_1 = nn.Linear(100, 10)
		self.layer_2 = nn.Linear(10, 1)
		
		# Sigmoid function for output
		self.sigmoid = nn.Sigmoid()

		# Initialize weights to 0
		#self._initialize_weights()

	def x_initialize_weights(self):
		nn.init.constant_(self.input_layer.weight, 0)
		nn.init.constant_(self.input_layer.bias, 0)
		nn.init.constant_(self.layer_1.weight, 0)
		nn.init.constant_(self.layer_1.bias, 0)

	def forward(self, x):
		x = torch.relu(self.input_layer(x))
		
		#x = self.layer_1(x)
		
		x = torch.relu(self.layer_1(x))
		x = self.layer_2(x)
		
		x = self.sigmoid(x)
		return x
"""

class EegNN(nn.Module):
	def __init__(self):
		super(EegNN, self).__init__()
		self.model = nn.Sequential(
			nn.Linear(20, 100),
			nn.ReLU(),
			nn.Linear(100, 1),
			nn.Sigmoid()
			)

	def forward(self, x):
		return self.model(x)

def GenerateData(inputs, labels):

	Delta_TP9_i = 0
	Delta_AF7_i = 1
	Delta_AF8_i = 2
	Delta_TP10_i = 3
	Theta_TP9_i = 4
	Theta_AF7_i = 5
	Theta_AF8_i = 6
	Theta_TP10_i = 7
	Alpha_TP9_i = 8
	Alpha_AF7_i = 9
	Alpha_AF8_i = 10
	Alpha_TP10_i = 11
	Beta_TP9_i = 12
	Beta_AF7_i = 13
	Beta_AF8_i = 14
	Beta_TP10_i = 15
	Gamma_TP9_i = 16
	Gamma_AF7_i = 17
	Gamma_AF8_i = 18
	Gamma_TP10_i = 19

	Delta_AF7_min = -0.5
	Delta_AF8_min = -0.5
	Delta_AF7_v = 0.0
	Delta_AF8_v = 0.0

	Delta_TP9_min = -0.1
	Delta_TP10_min = -0.1
	Delta_TP9_v = 0.2
	Delta_TP10_v = 0.2
	
	Theta_AF7_min = -0.5
	Theta_AF8_min = -0.5
	Theta_AF7_v = 0.0
	Theta_AF8_v = 0.0
	
	Theta_TP9_min = -0.1
	Theta_TP10_min = -0.1
	Theta_TP9_v = 0.2
	Theta_TP10_v = 0.2
	
	Alpha_AF7_min = 0.0
	Alpha_AF8_min = 0.0
	Alpha_AF7_v = 0.5
	Alpha_AF8_v = 0.5
	
	Alpha_TP9_min = 0.4
	Alpha_TP10_min = 0.4
	Alpha_TP9_v = 0.8
	Alpha_TP10_v = 0.8
	
	Beta_AF7_min = 0.0
	Beta_AF8_min = 0.0
	Beta_AF7_v = 1.4
	Beta_AF8_v = 1.4
	
	Beta_TP9_min = 0.0
	Beta_TP10_min = 0.0
	Beta_TP9_v = 0.5
	Beta_TP10_v = 0.5
	
	Gamma_AF7_min = 0.0
	Gamma_AF8_min = 0.0
	Gamma_AF7_v = 1.8
	Gamma_AF8_v = 1.8

	Gamma_TP9_min = 0.0
	Gamma_TP10_min = 0.0
	Gamma_TP9_v = 0.4
	Gamma_TP10_v = 0.4
	
	for p in range(100):
		inputs[p, Delta_TP9_i] = Delta_TP9_v if p % 2 == 0 else Delta_TP9_min
		inputs[p, Delta_AF7_i] = Delta_AF7_v if p % 2 == 0 else Delta_AF7_min
		inputs[p, Delta_AF8_i] = Delta_AF8_v if p % 2 == 0 else Delta_AF8_min
		inputs[p, Delta_TP10_i] = Delta_TP10_v if p % 2 == 0 else Delta_TP10_min
		inputs[p, Theta_TP9_i] = Theta_TP9_v if p % 2 == 0 else Theta_TP9_min
		inputs[p, Theta_AF7_i] = Theta_AF7_v if p % 2 == 0 else Theta_AF7_min
		inputs[p, Theta_AF8_i] = Theta_AF8_v if p % 2 == 0 else Theta_AF8_min
		inputs[p, Theta_TP10_i] = Theta_TP10_v if p % 2 == 0 else Theta_TP10_min
		inputs[p, Alpha_TP9_i] = Alpha_TP9_v if p % 2 == 0 else Alpha_TP9_min
		inputs[p, Alpha_AF7_i] = Alpha_AF7_v if p % 2 == 0 else Alpha_AF7_min
		inputs[p, Alpha_AF8_i] = Alpha_AF8_v if p % 2 == 0 else Alpha_AF8_min
		inputs[p, Alpha_TP10_i] = Alpha_TP10_v if p % 2 == 0 else Alpha_TP10_min

		inputs[p, Beta_TP9_i] = Beta_TP9_v * p / 100 if Beta_TP9_v * p / 100 > Beta_TP9_min else Beta_TP9_min
		inputs[p, Beta_AF7_i] = Beta_AF7_v * p / 100 if Beta_AF7_v * p / 100 > Beta_AF7_min else Beta_AF7_min
		inputs[p, Beta_AF8_i] = Beta_AF8_v * p / 100 if Beta_AF8_v * p / 100 > Beta_AF8_min else Beta_AF8_min
		inputs[p, Beta_TP10_i] = Beta_TP10_v * p / 100 if Beta_TP10_v * p / 100 > Beta_TP10_min else Beta_TP10_min
		inputs[p, Gamma_TP9_i] = Gamma_TP9_v * p / 100 if Gamma_TP9_v * p / 100 > Gamma_TP9_min else Gamma_TP9_min
		inputs[p, Gamma_AF7_i] = Gamma_AF7_v * p / 100 if Gamma_AF7_v * p / 100 > Gamma_AF7_min else Gamma_AF7_min
		inputs[p, Gamma_AF8_i] = Gamma_AF8_v * p / 100 if Gamma_AF8_v * p / 100 > Gamma_AF8_min else Gamma_AF8_min
		inputs[p, Gamma_TP10_i] = Gamma_TP10_v * p / 100 if Gamma_TP10_v * p / 100 > Gamma_TP10_min else Gamma_TP10_min

		labels[p] = p / 100

	#print()	

#['TimeStamp','Gyro_X','Gyro_Y','Gyro_Z','Accelerometer_X','Accelerometer_Y','Accelerometer_Z','AUX_RIGHT','HSI_TP9','HSI_AF7',
#'HSI_AF8','HSI_TP10','RAW_TP9','RAW_AF7','RAW_AF8','RAW_TP10','HeadBandOn','Battery','Elements','Mellow','Concentration']

filename = 'C:\\Users\\Steph\\Documents\\Yijing\\Muse\\2024-07-29-17-24-00-Muse.csv'
all_columns = ['Delta_TP9','Delta_AF7','Delta_AF8','Delta_TP10','Theta_TP9','Theta_AF7','Theta_AF8','Theta_TP10','Alpha_TP9',
    'Alpha_AF7','Alpha_AF8','Alpha_TP10','Beta_TP9','Beta_AF7','Beta_AF8','Beta_TP10','Gamma_TP9','Gamma_AF7','Gamma_AF8','Gamma_TP10']
tail = 11000

def train_eeg_model1():

	inputs = torch.zeros(100, 20, dtype=torch.float32, requires_grad=False).to(device)
	labels = torch.zeros(100, 1, dtype=torch.float32, requires_grad=False).to(device)
	GenerateData(inputs, labels)

	model = EegNN().to(device)

	# Define a loss function and an optimizer
	criterion = nn.BCELoss()  # The target scores run from 0 to 0.99.
	optimizer = optim.SGD(model.parameters(), lr=0.1)  # Stochastic Gradient Descent optimizer  0.001

	# Training loop
	num_epochs = 300_000
	for epoch in range(num_epochs):
		# Zero the gradients
		optimizer.zero_grad()
		# Forward pass: compute the model output
		outputs = model(inputs)
		# Compute the loss
		loss = criterion(outputs, labels)
		# Backward pass: compute gradients
		loss.backward()
		# Update weights
		optimizer.step()
		if (epoch+1) % 100_000 == 0:
			print(f'Epoch [{epoch+1}/{num_epochs}], Cost: {loss.item():.4f}')

	print()

	with open('eeg_model1_state.pt', 'wb') as f: 
		save(model.state_dict(), f) 

def test_eeg_model1(recording=None):
	if recording is None:
		inputs = torch.zeros(100, len(all_columns), dtype=torch.float32, device=device)
		labels = torch.zeros(100, 1, dtype=torch.float32, device=device)
		GenerateData(inputs, labels)
		title = 'Synthetic EEG predictions'
		output_file = 'Stephen V.csv'
	else:
		data = pd.read_csv(recording, usecols=all_columns).tail(tail).head(100)
		inputs = torch.from_numpy(data[all_columns].to_numpy(dtype=np.float32, copy=True)).to(device)
		title = 'Muse recording predictions'
		output_file = 'Stephen V.new.csv'

	model = EegNN().to(device)
	with open('eeg_model1_state.pt', 'rb') as f:
		model.load_state_dict(load(f, map_location=device, weights_only=True))
	model.eval()
	with torch.no_grad():
		predictions = model(inputs).squeeze(1).cpu().numpy()

	df2 = pd.DataFrame(inputs.cpu().numpy(), columns=all_columns)
	df2 = df2.rename(columns={df2.columns[0]: 'Prediction'})
	df2['Prediction'] = predictions
	df2.to_csv(output_file, index=False)

	df2.plot(kind='line', title=title, legend=True, figsize=(20, 10), linewidth=3.0)

# Train and check the fit on the generated data.
train_eeg_model1()
test_eeg_model1() # filename
# To score the Muse recording, use test_eeg_model1(filename).